In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install "protobuf>=5.26,<6" mediapipe -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 15.0 MB/s eta 0:00:00


In [ ]:
import scipy.io as sio
import glob, os, json
import pandas as pd
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!tar -xzf '/content/drive/MyDrive/Ketastasia/data/Penn_Action.tar.gz' -C '/content/'
!unzip -o -q '/content/drive/MyDrive/Ketastasia/data/kaggle.zip' -d '/content/Kaggle'
print("მზადაა ✅")

მზადაა ✅


In [ ]:
import urllib.request

model_path = "/content/pose_landmarker.task"
if not os.path.exists(model_path):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task",
        model_path
    )

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE
)
detector = vision.PoseLandmarker.create_from_options(options)
print("detector მზადაა ✅")

detector მზადაა ✅


In [ ]:
manifest_df = pd.read_csv('/content/drive/MyDrive/Ketastasia/data/combined.csv')
split_map = dict(zip(manifest_df['video_id'], manifest_df['split']))

penn_video_meta  = manifest_df[manifest_df['source'] == 'penn'].reset_index(drop=True)
kaggle_video_meta = manifest_df[manifest_df['source'] == 'kaggle'].reset_index(drop=True)

print(f"Penn videos:   {len(penn_video_meta)}")
print(f"Kaggle videos: {len(kaggle_video_meta)}")
print(manifest_df['split'].value_counts())

Penn videos:   1163
Kaggle videos: 650
split
train    1295
val       259
test      259
Name: count, dtype: int64


In [ ]:
print(penn_video_meta.columns.tolist())
print(penn_video_meta.head(2))

['video_id', 'source', 'label', 'split']
  video_id source        label  split
0     0341   penn  bench_press    val
1     0342   penn  bench_press  train


In [ ]:
FITNESS_ACTIONS = ['squat','pushup','pullup','jumping_jacks','situp',
                   'bench_press','jump_rope','clean_and_jerk']

penn_frame_rows = []

for _, vrow in penn_video_meta.iterrows():
    # mat_file path video_id-დან ავაწყოთ
    mat_file = f"/content/Penn_Action/labels/{vrow['video_id']}.mat"

    ann = sio.loadmat(mat_file)
    x, y = ann['x'], ann['y']

    for frame_idx in range(x.shape[0]):
        row = {
            'video_id': vrow['video_id'],
            'source': 'penn',
            'frame': frame_idx,
            'label': vrow['label']
        }
        for j in range(13):
            row[f'x{j}'] = x[frame_idx, j]
            row[f'y{j}'] = y[frame_idx, j]
        penn_frame_rows.append(row)

penn_df_13 = pd.DataFrame(penn_frame_rows)
penn_df_13['split'] = penn_df_13['video_id'].map(split_map)
print(f"Penn frames: {len(penn_df_13)}")
print(penn_df_13['split'].value_counts())

Penn frames: 96934
split
train    69004
test     14346
val      13584
Name: count, dtype: int64


In [ ]:
MEDIAPIPE_TO_PENN_ORDER = [0, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]

kaggle_frame_rows = []

for _, vrow in kaggle_video_meta.iterrows():
    # video_file path-ი ავაწყოთ video_id-დან
    # ვეძებთ ყველა ფოლდერში
    video_file = None
    for root, dirs, files in os.walk('/content/Kaggle'):
        for f in files:
            name = f.rsplit('.', 1)[0]
            if name == vrow['video_id']:
                video_file = os.path.join(root, f)
                break
        if video_file:
            break

    if not video_file:
        continue

    cap = cv2.VideoCapture(video_file)
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % 5 == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = detector.detect(mp_image)

            if result.pose_landmarks:
                lm = result.pose_landmarks[0]
                row = {
                    'video_id': vrow['video_id'],
                    'source': 'kaggle',
                    'frame': frame_idx,
                    'label': vrow['label']
                }
                for penn_j, mp_idx in enumerate(MEDIAPIPE_TO_PENN_ORDER):
                    row[f'x{penn_j}'] = lm[mp_idx].x
                    row[f'y{penn_j}'] = lm[mp_idx].y
                kaggle_frame_rows.append(row)

        frame_idx += 1
    cap.release()

kaggle_df_13 = pd.DataFrame(kaggle_frame_rows)
kaggle_df_13['split'] = kaggle_df_13['video_id'].map(split_map)
print(f"Kaggle frames: {len(kaggle_df_13)}")
print(kaggle_df_13['split'].value_counts())

Kaggle frames: 27138
split
train    19522
val       3939
test      3677
Name: count, dtype: int64


In [ ]:
combined_df_13 = pd.concat([penn_df_13, kaggle_df_13], ignore_index=True)

print(f"სულ frames: {len(combined_df_13)}")
print(combined_df_13.groupby('split')['label'].value_counts())

# Drive-ზე შენახვა
combined_df_13.to_csv(
    '/content/drive/MyDrive/Ketastasia/data/pipeline1A_13kp.csv',
    index=False
)
print("შენახულია ✅ pipeline1A_13kp.csv")

სულ frames: 124072
split  label              
test   clean_and_jerk         3355
       squat                  3157
       pullup                 2527
       bench_press            2008
       pushup                 1927
                              ... 
val    leg_extension           113
       leg_raises              112
       romanian_deadlift        92
       russian_twist            90
       decline_bench_press      51
Name: count, Length: 78, dtype: int64
შენახულია ✅ pipeline1A_13kp.csv
